In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Path to the unzipped model folder
MODEL_PATH = r"C:\Users\USER\Desktop\MediRAG\chunk-rag pipeline\flan_t5_neuro_rewriter" 

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)

# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model loaded on:", device)


c:\Users\USER\Desktop\MediRAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model loaded on: cpu


In [2]:
def rewrite_query(question: str) -> str:
    """
    Rewrite a medical question into a retrieval-optimized query.
    """
    prompt = "rewrite query: " + question

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=64,
            num_beams=4,
            early_stopping=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [3]:
question = (
    "What are some baseline clinical predictors of mortality in patients with intracranial hemorrhage?"
)

print("Original:", question)
print("Rewritten:", rewrite_query(question))


Original: What are some baseline clinical predictors of mortality in patients with intracranial hemorrhage?
Rewritten: baseline clinical predictors mortality intracranial hemorrhage
